In [1]:
import warnings
warnings.filterwarnings('ignore')

(Cookbook_Group_Free_Hierarchy)=
# Preserving molecules when adding groups

*Staging group hierarchy and using external selectors without changing molecular identity.*

Converted ligand graphs can have valid molecule membership with no groups or
chains. You can add groups through the native builder while retaining source
molecule indices and labels. MDTraj selections also work before groups are
added, using target-only placeholders rather than changing your native system.

:::{versionadded} 1.0.0
:::


## Starting with complete molecular graphs

The input has two connected molecules, a declared fixed chemical state and no
native groups. No coordinates or molecular preparation are needed.


In [2]:
import molsysmt as msm
from rdkit import Chem

molsys = msm.convert(Chem.MolFromSmiles('CCO.COC'), to_form='molsysmt.MolSys')
source_ids = molsys.topology.molecules.molecule_id.tolist()
print(msm.get(molsys, n_molecules=True, n_groups=True))


[2, 0]


## Selecting through MDTraj

MDTraj requires a residue and chain for every target atom. For missing native
groups, the adapter creates UNL placeholders per known molecule and chain, or
per atom when molecule membership is unknown. Missing chains use a fallback
chain. These parents exist only in the external target. UNL does not imply a
protein or water assignment. Selectors such as `resid` use this target metadata,
not missing native group indices. A declared group with conflicting known
chains is rejected rather than silently reassigned.


In [3]:
selected = msm.select(molsys, selection='index 0', syntax='MDTraj')
external = msm.convert(molsys, to_form='mdtraj.Topology')
print(selected)
print([[atom.index for atom in group.atoms] for group in external.residues])
print(msm.select(molsys, selection='protein', syntax='MDTraj'))
print(msm.get(molsys, n_groups=True, n_chains=True))


[0]
[[0, 1, 2], [3, 4, 5]]
[]
[0, 0]


## Adding groups to existing molecules

A source group-free molecule retains its identity when its atoms are assigned
to groups. The group names below are explicit illustrative labels. The builder
does not add duplicate molecule rows or reinterpret reference membership using
another chemical state's components. Source IDs remain string labels.


In [4]:
builder = msm.build.editable(molsys)
builder.add_group([0, 1, 2], group_name='LIG', group_id='ethanol')
builder.add_group([3, 4, 5], group_name='LIG', group_id='ether')
new_molsys = builder.build()
assert new_molsys.topology.molecules.molecule_id.tolist() == source_ids
print(msm.get(new_molsys, n_molecules=True, n_groups=True))
print(msm.get(new_molsys, element='molecule', atom_index=True))


[2, 2]
[[0, 1, 2], [3, 4, 5]]


## Completing a partial grouping

Building creates the ordinary fallback groups for remaining ungrouped source
atoms, while retaining their original molecule ownership. Repeated builds and
independent builder copies retain the same molecule axis.


In [5]:
partial_builder = msm.build.editable(molsys)
partial_builder.add_group([2], group_name='LIG')
partial = partial_builder.build()
print(msm.get(partial, n_molecules=True))
print(msm.get(partial, element='molecule', atom_index=True))


2
[[0, 1, 2], [3, 4, 5]]


## Rejecting conflicting groups

A group cannot span different inherited source molecules. Rejection occurs
before the group or atom memberships are changed. Explicit new systems assembled
without inherited molecule membership retain their ordinary builder inference.


In [6]:
builder = msm.build.editable(molsys)
try:
    builder.add_group([0, 3], group_name='LIG')
except msm.StructuralInconsistencyError:
    print('Cross-molecule grouping rejected; groups:', builder.n_groups)


Cross-molecule grouping rejected; groups: 0


## Calculating on the preserved molecules

Both the external selector and the compiled group hierarchy preserve access to
the complete source molecules. TPSA remains an area-valued graph descriptor;
the choice of selection syntax does not change its chemical model.


In [7]:
selected_area = msm.physchem.get_tpsa(
    molsys, selection='index 0', syntax='MDTraj',
)
areas = msm.physchem.get_tpsa(new_molsys)
print(msm.pyunitwizard.get_value(selected_area, to_unit='angstrom**2').round(2).tolist())
print(msm.pyunitwizard.get_value(areas, to_unit='angstrom**2').round(2).tolist())


[20.23]
[20.23, 9.23]


This workflow changes grouping, not the atom axis. Appending atoms to a builder
with populated chemical states has a separately reported limitation under
[uibcdf/molsysmt#499](https://github.com/uibcdf/molsysmt/issues/499). Native subset
extraction retains its existing order policy; use {func}`molsysmt.basic.permute_atoms`
when you explicitly need to reorder the complete atom axis before conversion.

:::{seealso}
:class: dropdown

- {ref}`Tutorial_Select` for native and external input syntaxes.
- {ref}`Tutorial_Editable` for staging and compiling systems.
- {ref}`Tutorial_Get_tpsa` for descriptor chemistry and units.
- {ref}`user-foundations-support-selection-syntaxes` for target metadata limits.
:::
